## 频域转化数据集

In [1]:
import numpy as np

def spectrum_like_model(X, modes=64):
    """
    对齐 SpectralConv1d 的频域处理
    输入:
        X: (B, L) 或 (B, C, L)
    输出:
        feat: (B, C, 2*modes)
    """

    # (B, L) → (B, 1, L)
    if X.ndim == 2:
        X = X[:, np.newaxis, :]

    # 1. cos（和模型一致）
    X = np.cos(X)

    # 2. rfft
    X_fft = np.fft.rfft(X, axis=-1)

    # 3. 截断低频
    X_fft = X_fft[:, :, :modes]

    # 4. amplitude + phase
    amp = np.abs(X_fft)
    phase = np.angle(X_fft)

    # 5. concat
    feat = np.concatenate([amp, phase], axis=-1)

    return feat


def process_dataset(input_path, output_path, modes=64):
    data = np.load(input_path)

    X = data['X']
    y = data['y']

    print(f"Processing {input_path} | shape={X.shape}")

    X_feat = spectrum_like_model(X, modes)

    print(f"Output shape: {X_feat.shape}")

    np.savez(output_path, X=X_feat, y=y)

In [3]:
process_dataset("./datasets/TemporalDrift/valid.npz",
                "./datasets/TemporalDrift/valid_fft.npz")

Processing ./datasets/TemporalDrift/valid.npz | shape=(2160, 10000)
Output shape: (2160, 1, 128)
